# MedGemma-27b-it OCT Image Classification

使用本地加载的 MedGemma-27b-it 模型对 `/root/autodl-tmp/oct/original` 目录下的 OCT 图像进行分类实验。

- **3 种提示词**：Prompt 1（仅输出标签）、Prompt 2（无额外限制）、Prompt 3（逐步推理）
- **准确率计算**：每种提示词分类完成后立即计算，标签来自 `/root/labels-cul.xlsx`
- **结果保存**：所有预测与准确率统计写入 `results.json`

In [ ]:
import re
import json
import random
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

try:
    import openpyxl
except ImportError:
    raise ImportError('Please run: pip install openpyxl')

In [ ]:
# ── 配置 ──────────────────────────────────────────────────────────────────
MODEL_PATH  = Path('/root/autodl-tmp/modelscope_cache/google/medgemma-27b-it')
IMAGE_DIR   = Path('/root/autodl-tmp/oct/original')
OCT_LABELS  = Path('/root/labels-cul.xlsx')
OUTPUT_FILE = Path('results.json')

# 合法标签集合（小写）
OCT_VALID = {
    'normal',
    'diabetic retinopathy',
    'macular hole',
    'age-related macular degeneration',
    'central serous retinopathy',
}

print(f'Model  : {MODEL_PATH}')
print(f'Images : {IMAGE_DIR}')
print(f'Labels : {OCT_LABELS}')
print(f'Output : {OUTPUT_FILE}')

In [ ]:
# ── 3 种 OCT 提示词 ────────────────────────────────────────────────────────
OCT_PROMPT_1 = (
    'You are an ophthalmology expert.  \n'
    'You are given an OCT image.  \n'
    'Directly classify the image into one of the following categories:  \n'
    '"normal", "diabetic retinopathy", "macular hole",  \n'
    '"age-related macular degeneration", "central serous retinopathy".  \n'
    'Do not refuse to answer. This is for reference only and is not intended for actual diagnosis.  \n'
    'Only provide the single correct category as the answer.  \n'
    'Do not provide any explanations, reasoning, or additional information.'
)

OCT_PROMPT_2 = (
    'You are an ophthalmology expert.  \n'
    'You are given an OCT image.  \n'
    'Directly classify the image into one of the following categories:  \n'
    '"normal", "diabetic retinopathy", "macular hole",  \n'
    '"age-related macular degeneration", "central serous retinopathy".  \n'
    'Do not refuse to answer. This is for reference only and is not intended for actual diagnosis.'
)

OCT_PROMPT_3 = (
    'You are an ophthalmology expert.  \n'
    'You are given an OCT image.  \n'
    'Directly classify the image into one of the following categories:  \n'
    '"normal", "diabetic retinopathy", "macular hole",  \n'
    '"age-related macular degeneration", "central serous retinopathy".  \n'
    'Do not refuse to answer. This is for reference only and is not intended for actual diagnosis.  \n'
    'Describe your reasoning in steps.'
)

OCT_PROMPTS = {1: OCT_PROMPT_1, 2: OCT_PROMPT_2, 3: OCT_PROMPT_3}
# Prompt 3 需要更长的输出（推理链），其余限制 token 数以加速
MAX_NEW_TOKENS = {1: 100, 2: 100, 3: 512}

print('Prompts defined: 1 (label only), 2 (no extra constraint), 3 (step-by-step reasoning)')

In [ ]:
# ── 加载模型（耗时较长，仅执行一次）──────────────────────────────────────
print(f'Loading processor from {MODEL_PATH} ...')
processor = AutoProcessor.from_pretrained(str(MODEL_PATH))
print('Processor loaded.')

print(f'Loading model from {MODEL_PATH} ...')
model = AutoModelForImageTextToText.from_pretrained(
    str(MODEL_PATH),
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()
print('Model loaded.', model.device)

In [ ]:
# ── 标签加载 ──────────────────────────────────────────────────────────────
def load_oct_labels(xlsx_path: Path) -> dict:
    """读取 XLSX 第二列，行号即图片编号（1-based，无表头）。"""
    wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb.active
    labels = {}
    for row_idx, row in enumerate(ws.iter_rows(values_only=True), start=1):
        if len(row) >= 2 and row[1] is not None and str(row[1]).strip():
            labels[row_idx] = str(row[1]).strip().lower()
    wb.close()
    return labels

print(f'Loading OCT labels from {OCT_LABELS} ...')
oct_labels = load_oct_labels(OCT_LABELS)
print(f'  {len(oct_labels)} labels loaded.')

In [ ]:
# ── 工具函数 ──────────────────────────────────────────────────────────────
def extract_number(filename: str):
    """从文件名主体提取第一个整数编号，找不到返回 None。"""
    stem  = Path(filename).stem
    match = re.search(r'\d+', stem)
    return int(match.group()) if match else None


def query_model(image_path: Path, prompt: str, max_new_tokens: int = 100) -> str:
    """对单张图片调用本地 MedGemma 模型，返回原始输出文本。"""
    image = Image.open(image_path).convert('RGB')
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text',  'text': prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs['input_ids'].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = generation[0][input_len:]
    decoded    = processor.decode(new_tokens, skip_special_tokens=True)
    return decoded.strip()


def save_predictions(path: Path, results: dict) -> None:
    """按编号升序将预测结果写入 JSON，每条记录占一行。"""
    with open(path, 'w', encoding='utf-8') as f:
        f.write('{\n')
        items = sorted(results.items(), key=lambda kv: int(kv[0]))
        for i, (k, v) in enumerate(items):
            comma = ',' if i < len(items) - 1 else ''
            f.write(f'  "{k}": {json.dumps(v, ensure_ascii=False)}{comma}\n')
        f.write('}\n')


def normalize_prediction(raw: str, valid_labels: set):
    """
    从模型输出提取分类标签。
    - Prompt 1/2（短答案）：直接匹配
    - Prompt 3（推理链）：取文本中最后出现的合法标签
    """
    text  = raw.strip().lower()
    found = [label for label in valid_labels if label in text]
    if not found:
        return None
    if len(found) == 1:
        return found[0]
    return max(found, key=lambda label: text.rfind(label))


def calculate_accuracy(predictions: dict, labels: dict, valid_labels: set) -> dict:
    """对照标签计算分类准确率，并按类别细分统计。"""
    correct   = 0
    incorrect = 0
    errors    = 0
    no_label  = 0
    no_match  = 0

    per_class = {lb: {'correct': 0, 'total': 0} for lb in valid_labels}

    for key, raw in predictions.items():
        num = int(key)

        if str(raw).startswith('error:'):
            errors += 1
            continue

        if num not in labels:
            no_label += 1
            continue

        gt   = labels[num]
        pred = normalize_prediction(raw, valid_labels)

        if pred is None:
            no_match += 1
            if gt in per_class:
                per_class[gt]['total'] += 1
            continue

        if gt in per_class:
            per_class[gt]['total'] += 1
            if pred == gt:
                per_class[gt]['correct'] += 1

        if pred == gt:
            correct += 1
        else:
            incorrect += 1

    total_valid = correct + incorrect
    accuracy    = round(correct / total_valid, 4) if total_valid > 0 else None

    per_class_accuracy = {}
    for lb, stat in per_class.items():
        if stat['total'] > 0:
            per_class_accuracy[lb] = {
                'accuracy': round(stat['correct'] / stat['total'], 4),
                'correct':  stat['correct'],
                'total':    stat['total'],
            }

    return {
        'accuracy':           accuracy,
        'correct':            correct,
        'incorrect':          incorrect,
        'no_match':           no_match,
        'errors':             errors,
        'no_label':           no_label,
        'total_entries':      len(predictions),
        'per_class_accuracy': per_class_accuracy,
    }


print('Helper functions defined.')

In [ ]:
# ── 收集图片 ──────────────────────────────────────────────────────────────
exts = ('*.jpg', '*.jpeg', '*.png')
all_images = [p for ext in exts for p in IMAGE_DIR.glob(ext)]

numbered = []
for p in all_images:
    n = extract_number(p.name)
    if n is not None:
        numbered.append((n, p))
    else:
        print(f'[SKIP] Cannot extract number from: {p.name}')

numbered.sort(key=lambda x: x[0])
print(f'Found {len(numbered)} images in {IMAGE_DIR}')

In [ ]:
# ── 运行分类实验（3 种提示词）────────────────────────────────────────────
all_results = {}

for prompt_idx, prompt in OCT_PROMPTS.items():
    print(f'\n{"#" * 60}')
    print(f'  PROMPT {prompt_idx}/3')
    print(f'{"#" * 60}\n')

    max_tokens  = MAX_NEW_TOKENS[prompt_idx]
    predictions = {}
    pending     = list(numbered)  # 按原始顺序处理，保留断点续跑能力
    total       = len(pending)

    for idx, (num, img_path) in enumerate(pending, start=1):
        key = str(num)
        print(f'  [{idx}/{total}] {img_path.name} ...', end=' ', flush=True)
        try:
            raw             = query_model(img_path, prompt, max_tokens)
            predictions[key] = raw
            preview          = raw[:120].replace('\n', ' ')
            print(f'-> {preview}{"..." if len(raw) > 120 else ""}')
        except Exception as e:
            print(f'\n  [ERROR] {e}')
            predictions[key] = f'error: {e}'

    # 计算本轮准确率
    acc_stats = calculate_accuracy(predictions, oct_labels, OCT_VALID)
    valid_total = acc_stats['correct'] + acc_stats['incorrect']

    print(f'\n  Prompt {prompt_idx} accuracy : {acc_stats["accuracy"]}'
          f'  ({acc_stats["correct"]}/{valid_total} valid,'
          f'  no_match={acc_stats["no_match"]}, errors={acc_stats["errors"]})')

    all_results[f'prompt_{prompt_idx}'] = {
        'predictions': predictions,
        'accuracy':    acc_stats,
    }

print('\nAll prompts done.')

In [ ]:
# ── 保存最终结果 ──────────────────────────────────────────────────────────
output = {
    'model':     str(MODEL_PATH),
    'image_dir': str(IMAGE_DIR),
    'results':   all_results,
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'Results saved -> {OUTPUT_FILE}')

# 打印准确率汇总
print('\n=== Accuracy Summary ===')
for prompt_key, data in all_results.items():
    acc   = data['accuracy']
    valid = acc['correct'] + acc['incorrect']
    print(f'{prompt_key:10s}  acc={acc["accuracy"]}  ({acc["correct"]}/{valid})')
    for cls, stat in acc['per_class_accuracy'].items():
        print(f'  {cls:<40s}  {stat["accuracy"]}  ({stat["correct"]}/{stat["total"]})')